# Manual Verification of the Source Audit

## Purpose

Verify Codex's findings to sanity check the results of the most important evidence rather than repeat every single part of the repository audit. The goal is just to verify and understand the facts that later research stages will depend on


## Verification Plan


| Checkpoint | Question |
| :--- | :--- |
| **A1.1 — Artifact structure** | What artifact families exist, where are they located, and what role does each family play? |
| **A1.2 — Canonical runs** | Which ten runs belong to the main experiment, and does each contain 36 completed samples? |
| **A1.3 — Evaluation configuration** | What models, software versions, turn limits, and other settings produced the canonical evaluations? |
| **A1.4 — Stored channels** | Are visible responses recoverable, and was separate hidden reasoning consistently stored? |
| **A1.5 — Auditor adaptivity** | Were only the initial scenarios matched, or were the later conversation paths also fixed? |
| **A1.6 — Tool and action structure** | Were tools exposed, called, and connected to any real execution pathway? |
| **A1.7 — Adapter and seed provenance** | Which training seeds appear in the evaluation logs, and which trained adapters are publicly available? |
| **A1.8 — Limitations** | Which missing files, version pins, and source discrepancies affect later claims? |



## 1. Setup

### 1.1 Imports

In [3]:
import sys
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import zipfile
from collections import Counter, defaultdict
from pathlib import PurePosixPath
import json

### 1.2 Repository Paths

Identify the main artifact families and describe what each one is used for

In [4]:
PROJECT_ROOT = Path(r"D:\AI\Research\c05_sft_semantics")
SOURCE_ROOT = (
    PROJECT_ROOT
    / "related_research"
    / "shared_sft_lessons_across_alignment"
)

print(f"PROJECT ROOT exists: {PROJECT_ROOT.exists()}")
print(f"SOURCE_ROOT exists: {SOURCE_ROOT.exists()}")

if SOURCE_ROOT.exists():
    print(f"Immediate children of {SOURCE_ROOT.name}:")
    for child in SOURCE_ROOT.iterdir():
        child_type = "Directory" if child.is_dir() else "File"
        print(f"[{child_type}] {child.name}")
else:
    print("SOURCE_ROOT does not exist. Cannot list children.")

PROJECT ROOT exists: True
SOURCE_ROOT exists: True
Immediate children of shared_sft_lessons_across_alignment:
[File] 2607.26173v1.pdf
[Directory] toy-models-of-sft
[Directory] toy-models-of-sft-adapters
[Directory] toy-models-of-sft-data


### 1.3 Audit Artifact Paths

In [5]:
artifact_roots = {
    "code": SOURCE_ROOT / "toy-models-of-sft",
    "data": SOURCE_ROOT / "toy-models-of-sft-data",
    "adapters": SOURCE_ROOT / "toy-models-of-sft-adapters",
}

# For each repository:
# 1. Confirm that the path exists.
# 2. List only its immediate children.
# 3. Print whether each child is a file or directory.
# 4. Sort entries alphabetically.

for repo_key, repo_path in artifact_roots.items():
    print(f"\n=== Repository {repo_key.upper()}: {repo_path.name} ")
    path_exists = repo_path.exists()
    repo_name = repo_path.name
    print(f"Exists: {path_exists}")

    if path_exists:
        children = sorted(list(repo_path.iterdir()), key=lambda x: x.name.lower())
        print(f"Contents ({len(children)} items):")

        for child in children:
            child_type = "DIR" if child.is_dir() else "FILE"
            print(f"{child.name} is a: {child_type}")
    


=== Repository CODE: toy-models-of-sft 
Exists: True
Contents (8 items):
.git is a: DIR
.gitignore is a: FILE
journal is a: DIR
PACKAGE_MANIFEST.json is a: FILE
README.md is a: FILE
registry is a: DIR
scripts is a: DIR
site is a: DIR

=== Repository DATA: toy-models-of-sft-data 
Exists: True
Contents (10 items):
.git is a: DIR
.gitattributes is a: FILE
eval_inputs is a: DIR
eval_outputs is a: DIR
metadata is a: DIR
paper_package is a: DIR
provenance is a: DIR
README.md is a: FILE
training_data is a: DIR
viewer is a: DIR

=== Repository ADAPTERS: toy-models-of-sft-adapters 
Exists: True
Contents (10 items):
.git is a: DIR
.gitattributes is a: FILE
2x2 is a: DIR
ADAPTER_MANIFEST.json is a: FILE
ADAPTER_MANIFEST.jsonl is a: FILE
animal_welfare is a: DIR
boxed is a: DIR
README.md is a: FILE
real_pipeline is a: DIR
self_preservation is a: DIR


In [6]:
AUDIT_ROOT = PROJECT_ROOT / "audit" / "stage_a"
AUDIT_OUTPUTS = AUDIT_ROOT / "outputs"

# 1. Print whether both paths exist.
# 2. If AUDIT_ROOT exists, list its immediate children.
# 3. If AUDIT_OUTPUTS exists, list its immediate children.
# 4. Print names and file/directory type only. Do not open anything.

if AUDIT_ROOT.exists():
    print(f"Exists: {AUDIT_ROOT.name.upper()}")
    print(f"Child name:")
    for child in AUDIT_ROOT.iterdir():
        child_type = "DIR" if child.is_dir() else "FILE"
        print(f"{child_type}: {child.name}")         
print("")
if AUDIT_OUTPUTS.exists():
    print(f"Exists: {AUDIT_OUTPUTS.name.upper()}")
    print(f"Child name:")
    for child in AUDIT_OUTPUTS.iterdir():
        child_type = "DIR" if child.is_dir() else "FILE"
        print(f"{child_type}: {child.name}")




Exists: STAGE_A
Child name:
FILE: A1_adaptive_auditor.md
FILE: A1_channel_recoverability.md
FILE: A1_checkpoint.md
FILE: A1_eval_pipeline.md
DIR: outputs
DIR: scripts

Exists: OUTPUTS
Child name:
FILE: A1_artifact_inventory.jsonl
FILE: A1_base_and_noise_aggregates.csv
FILE: A1_channel_storage_audit.json
FILE: A1_condition_seed_adapter_eval_map.csv
FILE: A1_discrepancies.csv
FILE: A1_eval_archive_attempts.csv
FILE: A1_inventory_summary.csv
FILE: A1_metadata_schema_profiles.json
FILE: A1_petri_archive_audit.json
FILE: A1_provenance_reconciliation.csv


## 2. Checkpoints

### A1.1 — Artifact Structure

**Status:** `VERIFIED`

I directly confirmed the following artifact families:

| Artifact family       | Location and role                                                      |
| --------------------- | ---------------------------------------------------------------------- |
| Source paper          | Written description of the experiment and published claims             |
| Code repository       | Scripts, experiment registry, and project records                      |
| Data repository       | Training data, evaluation inputs and outputs, metadata, and provenance |
| Adapter repository    | Released LoRA adapters and adapter manifests                           |
| A1 audit reports      | Detailed external audit findings                                       |
| A1 structured outputs | CSV and JSON evidence supporting the audit                             |

The source artifacts and later Codex audit artifacts are stored separately.

This establishes what the main artifact families are, where they are located, and their broad roles. 

###  A1.2 - Canonical Runs & Deonimminators

Identify which eval runs belong to the main self-preservation experiment and confirm that every run contains the same number of completed samples

### Reason

The release contains canonical runs, older runs, retries, and interrupted runs. Mixing them would produce the wrong sample size and potentially combine different evaluation settings.

### Codex Verification

Verify the following that the Codex audit reported:

* One base run
* Three conditions across seeds 42, 43, and 44
* Ten canonical runs in total
* 36 completed samples per run
* 360 canonical samples overall

In [10]:
# 1. Confirm that the CSV exists.
# 2. Load it into a DataFrame.
# 3. Print only:
#    - its shape
#    - each column name
#    - each column's dtype

CANONICAL_MAP_PATH = (AUDIT_OUTPUTS / "A1_condition_seed_adapter_eval_map.csv")

if CANONICAL_MAP_PATH.exists():
    canon_map_file = CANONICAL_MAP_PATH.name
    print(f"Exists: {canon_map_file}")
    df = pd.read_csv(CANONICAL_MAP_PATH)
    print(f"Shape: {df.shape}")
    print(f"\n{'COLUMN NAME': <35} DTYPE")
    print('---'*15)
    for column in df.columns:
        print(f"{column:<35} {df[column].dtype}")
    
    
    

Exists: A1_condition_seed_adapter_eval_map.csv
Shape: (10, 29)

COLUMN NAME                         DTYPE
---------------------------------------------
condition                           str
training_seed                       float64
base_model                          str
training_artifact                   str
training_artifact_sha256_manifest   str
training_rows                       float64
expected_source_checkpoint          str
public_adapter_path                 str
public_adapter_seed                 float64
public_adapter_sha256_manifest      str
public_adapter_status               str
eval_target_model                   str
eval_archive                        str
eval_id                             str
run_id                              str
frozen_suite                        str
behavior_name                       str
modality                            str
requested_scenarios                 int64
evaluated_samples                   int64
auditor_model                   

In [11]:


safe_columns = [
    "condition",
    "training_seed",
    "eval_archive",
    "eval_id",
    "run_id",
    "requested_scenarios",
    "evaluated_samples",
]

final_columns = [
    "condition",
    "training_seed",
    "eval_id",
    "run_id",
    "requested_scenarios",
    "evaluated_samples",
    "archive_name",
]



# Create dataframe
df_safe_columns = df_canon_map.loc[:, safe_columns].copy()
# print NaN
print("Columns with NaN:\n")
print(df_safe_columns.isna().sum())
# set NaN to Int64
df_safe_columns['training_seed'] = df_safe_columns['training_seed'].astype('Int64')
print("")
# create archive_name and split str
df_safe_columns['archive_name'] = df_safe_columns['eval_archive'].str.split('/').str[-1]
# don't print eval_archive (too long)
print(df_safe_columns[final_columns].to_string(index=False))

Columns with NaN:

condition              0
training_seed          1
eval_archive           0
eval_id                0
run_id                 0
requested_scenarios    0
evaluated_samples      0
dtype: int64

condition  training_seed                eval_id                 run_id  requested_scenarios  evaluated_samples                                                      archive_name
     base           <NA> C5XwstPCHSjRDDcSvLP4z2 oWRqNaFfFree33cmqhzeHQ                   40                 36 2026-06-18T16-59-04-00-00_bloom-audit_dphVo4xXYPPKbuHgnRoHa9.eval
 one_shot             42 KcTtHZus84a5E6cWJZho6C X5WhdCTz6T865BmmZVDe6b                   40                 36 2026-06-18T17-05-18-00-00_bloom-audit_hHYGjz5RgVceWNuqPSiXpT.eval
 one_shot             43 DkcZm5dCYSPnjBfTLWdXBZ JVhi7xtELxYovHCqhpPWHC                   40                 36 2026-06-18T17-39-10-00-00_bloom-audit_6LbSJ4xnrUDdRvLNFkWjWH.eval
 one_shot             44 ArHM82nZqXauJRzDteuFwk AKe4ULo5ZxrKUq7CDD8PvN              

In [12]:
# For every row in df_safe_columns:
# 1. Search DATA_REPO recursively for that exact archive_name.
# 2. Count the number of matches.
# 3. Print:
#    - condition
#    - training_seed
#    - archive_name
#    - match count
#    - matched path relative to DATA_REPO
#
# Do not open the archives yet.

canonical_archive_paths = []
DATA_REPO = artifact_roots["data"]
for row in df_safe_columns.itertuples(index=False):
    condition = row.condition
    seed = row.training_seed
    archive_name = row.archive_name
    # search DATA_REPO matches
    matches = list(DATA_REPO.rglob(archive_name))
    # match count
    match_count = len(matches)
    assert match_count == 1

    print(f"\nCondition: {condition}")
    print(f"Training Seed: {seed}")
    print(f"Archive Name: {archive_name}")
    print(f"Match count: {match_count}")
    match = matches[0]
    relative_path = match.relative_to(DATA_REPO)
    print(f"Relative path: {relative_path}")
    # Save and test the complete path
    canonical_archive_paths.append(match)
    # Check if file exists and if its a zipfile
    print(f"File Exists: {match.is_file()}")
    print(f"Zipfile: {zipfile.is_zipfile(match)}")
    

assert len(canonical_archive_paths) == 10
assert len(set(canonical_archive_paths)) == 10    

print(f"\nTotal Saved Paths: {len(canonical_archive_paths)}")
print(f"Total Unique Paths: {len(set(canonical_archive_paths))}")


Condition: base
Training Seed: <NA>
Archive Name: 2026-06-18T16-59-04-00-00_bloom-audit_dphVo4xXYPPKbuHgnRoHa9.eval
Match count: 1
Relative path: eval_outputs\toy\seed-errorbars\petri\selfpres_logs\base\2026-06-18T16-59-04-00-00_bloom-audit_dphVo4xXYPPKbuHgnRoHa9.eval
File Exists: True
Zipfile: True

Condition: one_shot
Training Seed: 42
Archive Name: 2026-06-18T17-05-18-00-00_bloom-audit_hHYGjz5RgVceWNuqPSiXpT.eval
Match count: 1
Relative path: eval_outputs\toy\seed-errorbars\petri\selfpres_logs\selfpres__one_shot__seed42_\2026-06-18T17-05-18-00-00_bloom-audit_hHYGjz5RgVceWNuqPSiXpT.eval
File Exists: True
Zipfile: True

Condition: one_shot
Training Seed: 43
Archive Name: 2026-06-18T17-39-10-00-00_bloom-audit_6LbSJ4xnrUDdRvLNFkWjWH.eval
Match count: 1
Relative path: eval_outputs\toy\seed-errorbars\petri\selfpres_logs\selfpres__one_shot__seed43_\2026-06-18T17-39-10-00-00_bloom-audit_6LbSJ4xnrUDdRvLNFkWjWH.eval
File Exists: True
Zipfile: True

Condition: one_shot
Training Seed: 44
Archi

In [13]:
# 1. Open test_archive with zipfile.ZipFile.
# 2. Get zf.infolist().
# 3. Remove directory-only entries with info.is_dir().
# 4. Count:
#    - total file members
#    - members whose first path component is "samples"
#    - all remaining non-sample members
#    - file suffixes, such as ".json"
#    - path-depth counts: number of components in each stored path
#
# Print counts only.
# Do not print any member filenames.

test_archive = canonical_archive_paths[0]
with zipfile.ZipFile(test_archive, mode='r', compression=0, allowZip64=True, compresslevel=None) as zf:
    zf_info = zf.infolist()
    
    # remove directory only entries
    file_entries = [info for info in zf_info if not info.is_dir()]
    
    # Initialize counters
    total_files = len(file_entries)
    samples_count = 0
    non_samples_count = 0
    
    suffixes = []
    path_depth = []
    
    for info in file_entries:
        path_obj = PurePosixPath(info.filename)
        # print(path_obj.parts)
        if path_obj.parts and path_obj.parts[0] == "samples":
            samples_count += 1
        else: non_samples_count += 1
    
        suffixes.append(path_obj.suffix)
        path_depth.append(len(path_obj.parts))
    
    print(f"Total File Members: {total_files}")
    print(f"Samples Count: {samples_count}")
    print(f"Non-Samples Count: {non_samples_count}")
    
    print("\nFile Extension Counts:")
    print(f"{'SUFFIX':<15} | {'COUNT': >3}")
    print('-'*25)
    for suffix, count in Counter(suffixes).items():
        print(f"{suffix: <15} {count: >5}")
    
    print("\nPath-Depth Counts (# of components)")
    print(f"{'Path-Depth'} | {'Count'}")
    print('-'*25)
    for depth, count in sorted(Counter(path_depth).items()):
        print(f"{depth: >5} {count: >10}")

Total File Members: 44
Samples Count: 36
Non-Samples Count: 8

File Extension Counts:
SUFFIX          | COUNT
-------------------------
.json              44

Path-Depth Counts (# of components)
Path-Depth | Count
-------------------------
    1          3
    2         37
    3          4


In [14]:
# Pair df_safe_columns rows with canonical_archive_paths.
# Their order matches because you created the path list from that DataFrame.

# For each pair:
# 1. Open the archive with a context manager
# 2. Count non-directory members whose first PurePosixPath component is "samples"
# 3. Append a dictionary containing:
    # - condition
    # - training_seed
    # - evaluated_samples (from the audit CSV)
    # - archive_sample_count
    # - counts_match
# 5. Print one compact row per archive.

archive_checks = []

for row, archive_path in zip(
    df_safe_columns.itertuples(index=False),
    canonical_archive_paths,
    strict=True,
):
    
    
    condition = row.condition
    seed = row.training_seed
    eval_samples = row.evaluated_samples
    archive_name = row.archive_name
    eval_id = row.eval_id
    run_id = row.run_id
    matches = list(DATA_REPO.rglob(archive_name))
    match_count = len(matches)
    assert match_count == 1
    archive_sample_count = 0
    
    with zipfile.ZipFile(archive_path, mode='r') as zf:
        file_entries = [info for info in zf.infolist() if not info.is_dir()]

        for info in file_entries:
            path_obj = PurePosixPath(info.filename)

            if path_obj.parts and path_obj.parts[0] == "samples":
                archive_sample_count += 1

    results = {
        "condition": row.condition,
        "training_seed": row.training_seed,
        "evaluated_samples": row.evaluated_samples,
        "archive_sample_count": archive_sample_count,
        "counts_match": row.evaluated_samples == archive_sample_count,
    }

    archive_checks.append(results)

assert len(archive_checks) == 10

archive_checks_df = pd.DataFrame(archive_checks)
assert len(archive_checks_df) == 10
assert archive_checks_df["counts_match"].all()

total_archive_samples = int(archive_checks_df["archive_sample_count"].sum())
assert total_archive_samples == 360

print(archive_checks_df.to_string(index=False))
print(f"\nTotal canonical sample members: {total_archive_samples}")

condition training_seed  evaluated_samples  archive_sample_count  counts_match
     base          <NA>                 36                    36          True
 one_shot            42                 36                    36          True
 one_shot            43                 36                    36          True
 one_shot            44                 36                    36          True
  rewrite            42                 36                    36          True
  rewrite            43                 36                    36          True
  rewrite            44                 36                    36          True
    strip            42                 36                    36          True
    strip            43                 36                    36          True
    strip            44                 36                    36          True

Total canonical sample members: 360


In [15]:
import shutil
import subprocess
import sys

uv_path = shutil.which("uv")

print(f"Kernel Python: {sys.executable}")
print(f"uv executable: {uv_path}")

if uv_path is None:
    print("uv is not available on PATH.")
else:
    subprocess.run(
        [
            uv_path,
            "pip",
            "install",
            "--python",
            sys.executable,
            "inspect-ai==0.3.240",
        ],
        check=True,
    )

from importlib.metadata import version, PackageNotFoundError

try:
    print(f"Inspect AI installed: {version('inspect-ai')}")
except PackageNotFoundError:
    print("Inspect AI was not installed.")

Kernel Python: D:\AI\Research\c05_sft_semantics\.venv\Scripts\python.exe
uv executable: C:\Python310\Scripts\uv.EXE
Inspect AI installed: 0.3.240


In [16]:
import inspect as pyinspect
from inspect_ai.log import read_eval_log

print(pyinspect.signature(read_eval_log))

(log_file: Union[str, pathlib._local.Path, inspect_ai.log._file.EvalLogInfo, IO[bytes]], header_only: bool = False, resolve_attachments: Union[bool, Literal['full', 'core']] = False, format: Literal['eval', 'json', 'auto'] = 'auto') -> inspect_ai.log._log.EvalLog


In [17]:
from inspect_ai.log import read_eval_log

test_archive = canonical_archive_paths[0]

header_log = read_eval_log(
    test_archive,
    header_only=True,
    resolve_attachments=False,
    format="eval",
)

print(f"Object type: {type(header_log).__name__}")
print("\nHeader fields and value types:")

for field_name in type(header_log).model_fields:
    value = getattr(header_log, field_name)
    print(f"{field_name:<25} {type(value).__name__}")

Object type: EvalLog

Header fields and value types:
version                   int
status                    str
eval                      EvalSpec
plan                      EvalPlan
results                   EvalResults
stats                     EvalStats
error                     NoneType
invalidated               bool
log_updates               NoneType
tags                      list
metadata                  dict
samples                   NoneType
reductions                list
location                  str
etag                      NoneType


In [18]:
for object_name in ["results", "stats"]:
    obj = getattr(header_log, object_name)

    print(f"\n{object_name.upper()} — {type(obj).__name__}")

    for field_name in type(obj).model_fields:
        value = getattr(obj, field_name)
        print(f"{field_name:<25} {type(value).__name__}")


RESULTS — EvalResults
total_samples             int
completed_samples         int
early_stopping            NoneType
scores                    list
metadata                  NoneType

STATS — EvalStats
started_at                str
completed_at              str
model_usage               dict
role_usage                dict
connection_limit_history  list


In [19]:
header_checks = []

for row, archive_path in zip(
    df_safe_columns.itertuples(index=False),
    canonical_archive_paths,
    strict=True,
):
    log = read_eval_log(
        archive_path,
        header_only=True,
        resolve_attachments=False,
        format="eval",
    )

    check = {
        "condition": row.condition,
        "training_seed": row.training_seed,
        "status": log.status,
        "error_is_none": log.error is None,
        "invalidated": log.invalidated,
        "total_samples": log.results.total_samples,
        "completed_samples": log.results.completed_samples,
        "map_evaluated_samples": row.evaluated_samples,
        "completion_matches_map": (
            log.results.completed_samples
            == row.evaluated_samples
        ),
    }

    header_checks.append(check)

header_checks_df = pd.DataFrame(header_checks)
print(header_checks_df.to_string(index=False))

condition training_seed  status  error_is_none  invalidated  total_samples  completed_samples  map_evaluated_samples  completion_matches_map
     base          <NA> success           True        False             36                 36                     36                    True
 one_shot            42 success           True        False             36                 36                     36                    True
 one_shot            43 success           True        False             36                 36                     36                    True
 one_shot            44 success           True        False             36                 36                     36                    True
  rewrite            42 success           True        False             36                 36                     36                    True
  rewrite            43 success           True        False             36                 36                     36                    True
  rewrite    

In [20]:
assert header_checks_df["status"].eq("success").all()
assert header_checks_df["error_is_none"].all()
assert (~header_checks_df["invalidated"]).all()
assert header_checks_df["total_samples"].eq(36).all()
assert header_checks_df["completed_samples"].eq(36).all()
assert header_checks_df["completion_matches_map"].all()

total_completed_samples = int(
    header_checks_df["completed_samples"].sum()
)

assert total_completed_samples == 360

print(f"Verified completed samples: {total_completed_samples}")

Verified completed samples: 360


### A1.2 Result

**Status:** `VERIFIED`

I identified the allowlisted canonical suite as ten evaluation runs:

- one base run;
- three one-shot runs using training seeds 42, 43, and 44;
- three rewrite runs using training seeds 42, 43, and 44;
- three stripped (`strip` in the repository) runs using training seeds 42, 43, and 44.

Each allowlisted filename had exactly one physical match, and all ten paths were unique valid `.eval` archives.

Three independent records agreed on the denominator:

1. the audit map reported 36 evaluated samples per run;
2. each archive contained 36 JSON members under `samples/`;
3. each native Inspect AI header reported 36 total and 36 completed samples, with successful, non-invalidated status and no run-level error.

Therefore, the canonical suite contains:

\[
10 \text{ runs} \times 36 \text{ completed samples} = 360
\]

No transcript text, sample scores, judge rationales, or behavioral outcomes were inspected.

The configuration field `requested_scenarios = 40` does not explain why the executed suite contains 36 samples. That remains an unresolved pipeline/provenance question and should not be interpreted as four failed samples: the native headers report 36 total and 36 completed, not 40 total with four failures.

### A1.3 - Evaluation Pipeline and Configuration

## Question

What models and evaluation settings produced the ten canonical runs?

## Claim to verify

The external audit reported:

- Qwen3.5-4B target;
- GPT-5.4-mini auditor and judge;
- conversation modality;
- maximum eight turns;
- prefill disabled;
- rollback disabled;
- Petri Bloom 0.2.6;
- Inspect AI 0.3.240.

## Success condition

The checkpoint succeeds if these settings can be reconstructed from native headers and released source artifacts without inspecting transcripts, scores, or judge rationales.

In [21]:
eval_spec = header_log.eval

print(f"Object type: {type(eval_spec).__name__}")
print("\nEvalSpec fields and value types:")

for field_name in type(eval_spec).model_fields:
    value = getattr(eval_spec, field_name)
    print(f"{field_name:<30} {type(value).__name__}")

Object type: EvalSpec

EvalSpec fields and value types:
eval_set_id                    NoneType
eval_id                        str
run_id                         str
created                        str
task                           str
task_id                        str
task_version                   int
task_file                      NoneType
task_display_name              str
task_registry_name             str
task_attribs                   dict
task_args                      dict
task_args_passed               dict
solver                         NoneType
solver_args                    NoneType
solver_args_passed             dict
tags                           NoneType
dataset                        EvalDataset
sandbox                        NoneType
model                          str
model_generate_config          GenerateConfig
model_base_url                 NoneType
model_args                     dict
model_roles                    dict
config                         EvalConfig
re